# Critic / Reflection Mode

Critic/reflection mode is a behavioral pattern where the agent's primary activity is evaluating and improving an existing output — its own or another agent's. Rather than producing content from scratch, the agent reads, critiques, identifies weaknesses, and proposes revisions.

This mode implements the maker-checker pattern: one agent (or the same agent in a prior turn) produces; the critic evaluates and revises. The separation between making and checking is what makes the pattern valuable — the critic adopts a different perspective than the maker, surfacing issues the maker would miss.

Critic/reflection mode is classified as self-evaluation and revision. It can combine with any autonomy level:
- Critic + copilot: Agent provides critique, human makes revisions.
- Critic + supervised: Agent revises, human approves the revised version.
- Critic + fully autonomous: Agent critiques and revises iteratively until a quality threshold is met.

In [1]:
import os
import json
from dataclasses import dataclass
from typing import Optional

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

### Initialize the language model
Critic/reflection mode uses `temperature=0` for consistent, reproducible evaluations. When scoring content against criteria, variation between runs is a liability — two evaluations of the same content should produce the same quality scores, not different assessments. Determinism here is what makes the quality threshold meaningful as a repeatable pass/fail signal.

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY", "").strip())

print("LLM initialized for critic/reflection mode.")

LLM initialized for critic/reflection mode.


## Defining the critique data model
Before building the critic agent, we need a clear schema for what a critique actually contains. The model captures three distinct things: individual issues found in the content, per-criterion quality scores, and enough metadata to determine whether the content is ready for handoff.

`CritiqueIssue` forces specificity. Vague criticism — "this section is unclear" — is not actionable. By requiring a `location`, `issue`, `severity`, and `suggestion`, the model prevents the critic from producing feedback that the maker cannot act on. The `suggestion` field is the most important: it must propose specific corrective language, not just describe what is wrong.

`CritiqueResult` is the session-level record. It stores the original content (so revisions can always be diff'd against the source), all issues found across iterations, the revised version, per-criterion scores, and the `ready_for_delivery` flag — a boolean contract that downstream systems can check without parsing any prose.

In [3]:
@dataclass
class CritiqueIssue:
    """A single issue identified during critique."""
    criterion: str    # which quality criterion this issue relates to
    issue: str        # what specifically is wrong
    location: str     # where in the content (section, line, or phrase)
    severity: str     # 'critical' | 'major' | 'minor'
    suggestion: str   # exact corrective language or approach


@dataclass
class CritiqueResult:
    """The result of one or more critique/revision passes."""
    original: str              # content submitted for review — kept for diffing
    issues_found: list         # all CritiqueIssue objects across all iterations
    revised: Optional[str]     # revised version (None if original already met the threshold)
    criteria_scores: dict      # criterion text → score (0.0–1.0)
    overall_score: float       # average across all criteria scores
    iteration: int             # which revision cycle produced this result
    ready_for_delivery: bool   # True when overall_score >= quality_threshold
    quality_notes: str         # summary of remaining gaps or strengths


print("Critique data model defined.")
print(f"  CritiqueIssue fields : {list(CritiqueIssue.__dataclass_fields__.keys())}")
print(f"  CritiqueResult fields: {list(CritiqueResult.__dataclass_fields__.keys())}")

Critique data model defined.
  CritiqueIssue fields : ['criterion', 'issue', 'location', 'severity', 'suggestion']
  CritiqueResult fields: ['original', 'issues_found', 'revised', 'criteria_scores', 'overall_score', 'iteration', 'ready_for_delivery', 'quality_notes']


`CritiqueIssue` and `CritiqueResult` serve different layers of the pipeline. `CritiqueIssue` holds one identified problem at an atomic level — the `location` and `suggestion` fields are the ones that make a critique actionable. Without a precise location an executor cannot find what to fix; without a concrete suggestion they cannot determine what the correct fix looks like.

The `severity` field drives prioritization during revision. A `critical` issue, if unresolved, would lead a reviewer to reject the content entirely. `major` issues are important but survivable; `minor` issues are polish. The revision function uses this field to filter issues before passing them to the editor model, ensuring the most important problems are addressed first.

`overall_score` is stored as a pre-computed float rounded to two decimal places. Storing it separately from `criteria_scores` means callers can threshold-check it without recomputing the average. The `ready_for_delivery` flag is a boolean contract: downstream systems read this field directly rather than parsing quality notes.

## Defining quality criteria
Quality criteria are the foundation of critic mode. They define what "good" means for this specific content in this specific context. Before any critique begins, the criteria must be explicit and agreed upon — allowing them to change mid-review is one of the most common anti-patterns in critique workflows, because it retroactively invalidates earlier assessments.

We will use a concrete example throughout this notebook: reviewing API documentation for an external REST API. The five criteria below reflect what good API documentation must achieve. Each criterion is specific enough to be evaluable: "At least one concrete request and response example is provided" is testable; "Is well-written" is not. The draft content is intentionally incomplete — it has the correct structure but omits parameter types, examples, and error documentation — because realistic quality failures come from partial content, not from content that is entirely wrong.

In [4]:
# Quality criteria for API documentation — each criterion is independently evaluable
QUALITY_CRITERIA = [
    "Accurately describes what the endpoint does and what it returns",
    "All parameters are documented with types, required/optional status, and valid values",
    "At least one concrete request and response example is provided",
    "Error cases and their status codes are documented",
    "Is clearly understandable by a developer unfamiliar with the codebase",
]

# Draft content intentionally omits parameter types, examples, and error docs
DRAFT_CONTENT = """
## POST /api/v1/orders

Creates a new order in the system.

### Request
Send a JSON body with the order details. The `customer_id` field is required.
You can also include `items` which is a list of products.

### Response
Returns the created order object with an `id` field.

### Notes
Make sure your request is valid or you will get an error.
Authentication is required.
"""

print("Quality criteria:")
for i, criterion in enumerate(QUALITY_CRITERIA, 1):
    print(f"  {i}. {criterion}")

print("\nDraft content to review:")
print("-" * 60)
print(DRAFT_CONTENT)

Quality criteria:
  1. Accurately describes what the endpoint does and what it returns
  2. All parameters are documented with types, required/optional status, and valid values
  3. At least one concrete request and response example is provided
  4. Error cases and their status codes are documented
  5. Is clearly understandable by a developer unfamiliar with the codebase

Draft content to review:
------------------------------------------------------------

## POST /api/v1/orders

Creates a new order in the system.

### Request
Send a JSON body with the order details. The `customer_id` field is required.
You can also include `items` which is a list of products.

### Response
Returns the created order object with an `id` field.

### Notes
Make sure your request is valid or you will get an error.
Authentication is required.



The five criteria span independent quality dimensions: correctness, completeness, concreteness, safety coverage, and accessibility. A document can score high on correctness while failing on accessibility, and per-criterion scoring captures that distinction — it prevents a single strong dimension from masking failures in others.

`DRAFT_CONTENT` is a module-level constant defined alongside `QUALITY_CRITERIA` because both are fixed inputs that the evaluation, revision, and pipeline functions all need to reference. Defining them once at the top prevents the per-cell repetition that would occur if each function call had its own inline content string.

## Step 1: Evaluating content against criteria
The evaluation step is the core of critic mode. The agent reads the content, scores it against each criterion on a 0–1 scale, and produces a list of specific issues with exact locations and actionable suggestions.

The prompt structure is deliberate: the context is listed first, then the criteria, then the content. This ordering ensures the model reads the intended use case before seeing either the criteria or the content, which anchors evaluation to the right frame of reference. Criteria are listed before content so the model knows what to look for before it reads — otherwise it forms an overall impression first and reverse-engineers criterion scores to match it.

The `top_priorities` field in the response limits the revision step to the most impactful issues. A critique that identifies fifteen problems is technically thorough but practically overwhelming. Identifying the three most critical problems and fixing those first is more effective than attempting to fix all fifteen in one pass, which tends to introduce new issues while solving old ones.

In [5]:
def evaluate_content(content: str, criteria: list, context: str = "", iteration: int = 1) -> dict:
    """Evaluate content against each quality criterion and identify specific issues.

    Args:
        content: The content to evaluate.
        criteria: List of quality criteria to evaluate against.
        context: What the content is supposed to accomplish.
        iteration: Which revision cycle this is (included in the prompt for tracking).

    Returns:
        Dict with 'criteria_scores', 'issues', 'top_priorities', and 'overall_assessment'.
    """
    # Number each criterion so the model evaluates them in order, one at a time
    criteria_formatted = "\n".join(
        f"{i+1}. {criterion}" for i, criterion in enumerate(criteria)
    )

    # The system prompt defines the reviewer role and the exact JSON schema expected in the response
    system_prompt = """You are a precise quality reviewer. Evaluate the content against each criterion independently.

For each criterion:
- Score it from 0.0 (completely fails) to 1.0 (fully meets)
- Identify specific issues with exact location in the text
- Propose specific corrective language

Respond with JSON only:
{
  "criteria_scores": {"criterion text": score},
  "issues": [
    {
      "criterion": "...",
      "issue": "what is wrong",
      "location": "where in the text",
      "severity": "critical | major | minor",
      "suggestion": "exact corrective language or approach"
    }
  ],
  "top_priorities": ["issue 1", "issue 2", "issue 3"],
  "overall_assessment": "brief summary"
}"""

    # Context is injected before the criteria so the model reads the use-case frame first
    prompt = f"""Evaluation iteration: {iteration}

Context (what this content must achieve):
{context or 'Not specified — evaluate based on general quality for this content type.'}

Quality criteria:
{criteria_formatted}

Content to evaluate:
{content}"""

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])

    try:
        return json.loads(response.content)
    except json.JSONDecodeError:
        # Strip markdown fences if the model wrapped the JSON in a code block
        raw = response.content.strip()
        if raw.startswith("```"):
            raw = "\n".join(raw.split("\n")[1:-1])
        return json.loads(raw)

In [6]:
CONTEXT = "API documentation for a backend REST API, intended for external developers integrating with our platform."

# Run the first evaluation pass on the draft content
evaluation = evaluate_content(DRAFT_CONTENT, QUALITY_CRITERIA, CONTEXT, iteration=1)

# Compute overall score as the mean of all per-criterion scores
scores = evaluation["criteria_scores"]
overall = sum(scores.values()) / len(scores)

print(f"Initial evaluation — overall score: {overall:.2f} / 1.00")
print("-" * 60)
print("Per-criterion scores:")
for criterion, score in scores.items():
    icon = "✓" if score >= 0.7 else "✗"   # threshold for individual criterion pass
    short = criterion[:55] + "..." if len(criterion) > 55 else criterion
    print(f"  {icon} {short}: {score:.2f}")

print(f"\nIssues found: {len(evaluation['issues'])}")
print("-" * 60)
for issue in evaluation["issues"]:
    print(f"  [{issue['severity'].upper()}] {issue['issue']}")
    print(f"           Location  : {issue['location']}")
    print(f"           Suggestion: {issue['suggestion']}")
    print()

print("Top priorities:")
for i, p in enumerate(evaluation["top_priorities"], 1):
    print(f"  {i}. {p}")

Initial evaluation — overall score: 0.36 / 1.00
------------------------------------------------------------
Per-criterion scores:
  ✓ Accurately describes what the endpoint does and what it...: 0.80
  ✗ All parameters are documented with types, required/opti...: 0.40
  ✗ At least one concrete request and response example is p...: 0.00
  ✗ Error cases and their status codes are documented: 0.00
  ✗ Is clearly understandable by a developer unfamiliar wit...: 0.60

Issues found: 4
------------------------------------------------------------
  [MAJOR] Parameters lack detailed documentation including types and valid values.
           Location  : Request section
           Suggestion: Document parameters as follows: `customer_id` (string, required) - The ID of the customer. `items` (array of objects, optional) - List of products, each with `product_id` (string) and `quantity` (integer).

  [CRITICAL] No examples of request and response are provided.
           Location  : Response section


`criteria_formatted` is constructed with `enumerate` to produce numbered labels (`"1. criterion text"`). Numbering gives the model a positional reference for each criterion that appears in the `criteria_scores` dict keys, making it easy to match scores back to criteria by index if needed.

The `iteration` argument is injected at the top of the user message as `"Evaluation iteration: N"`. This is not used by the parsing logic — it is there for the model, which produces more focused evaluations on later iterations when it knows a previous pass has already been attempted. The JSON schema in the system prompt specifies `"severity": "critical | major | minor"` as a fixed enum; using an explicit enum means the revision function can filter issues by severity reliably, without fuzzy string matching.

## Step 2: Targeted revision
Revision in critic mode is deliberately constrained. The revision prompt includes only the top-priority issues — filtered to `critical` and `major` severity and capped at three — not the full issue list. This constraint is intentional: attempting to fix many issues simultaneously tends to produce rewrites that address surface problems while introducing new structural ones.

The key instruction — "fix exactly what is wrong; do not rewrite sections that are not flagged" — works against the LLM's default behavior of producing thorough rewrites. Without this constraint, the model treats the critique as permission to improve anything it considers suboptimal, a tendency that produces content changed beyond what the critique authorized. The revised content is returned as plain text with no explanation: the change is the record, not a summary of the change. The next evaluation pass measures whether the revision actually improved the scores.

In [7]:
def revise_content(content: str, evaluation: dict, context: str = "") -> str:
    """Revise content to address the top-priority issues from an evaluation.

    Args:
        content: The content to revise.
        evaluation: The evaluation dict from evaluate_content.
        context: What the content is supposed to accomplish.

    Returns:
        The revised content as a plain string.
    """
    # Filter to critical and major issues only — minor issues are deferred to later passes
    detailed_issues = [
        issue for issue in evaluation.get("issues", [])
        if issue["severity"] in ("critical", "major")
    ][:3]  # cap at 3 to keep each revision pass focused on one clear set of changes

    # The system prompt enforces surgical edits, preventing scope creep in the revision
    system_prompt = """You are a precise editor. Revise the content to address the listed issues.

RULES:
- Fix exactly what is wrong; do not rewrite sections that are not flagged
- Maintain the original structure, intent, and tone
- Each fix must directly address the stated issue using the provided suggestion
- Do not add new content beyond what is needed to fix the identified issues
- Return the revised content only — no explanation, no commentary"""

    prompt = f"""Context: {context}

Issues to fix:
{json.dumps(detailed_issues, indent=2)}

Original content:
{content}

Revise the content to fix these specific issues. Return the revised content only."""

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])
    return response.content.strip()

In [8]:
# Run one revision pass using the issues found in the initial evaluation
revised_content = revise_content(DRAFT_CONTENT, evaluation, CONTEXT)

print("Revised content:")
print("=" * 60)
print(revised_content)

Revised content:
## POST /api/v1/orders

Creates a new order in the system.

### Request
Send a JSON body with the order details. The `customer_id` (string, required) - The ID of the customer. You can also include `items` (array of objects, optional) - List of products, each with `product_id` (string) and `quantity` (integer).

### Response
Returns the created order object with an `id` field.

**Example Request:** `{ "customer_id": "12345", "items": [{ "product_id": "abc", "quantity": 2 }] }`  
**Example Response:** `{ "id": "67890", "customer_id": "12345", "items": [{ "product_id": "abc", "quantity": 2 }] }`

### Notes
Make sure your request is valid or you will get an error.  
Authentication is required.  
Error cases: `400 Bad Request` for invalid input, `401 Unauthorized` for authentication issues.


The `detailed_issues[:3]` slice applies two filters sequentially. First, the list comprehension retains only issues with `severity` in `("critical", "major")`, discarding minor polish issues that do not need to be addressed in the current pass. Second, the `[:3]` slice further caps the list at three items. Passing a long issue list to the editor model tends to produce revisions that partially address many issues rather than fully resolving a few — the cap enforces depth over breadth.

The issues are serialized as `json.dumps(detailed_issues, indent=2)` rather than being formatted as prose. JSON formatting preserves the exact field names (`location`, `suggestion`) that the editor needs to locate and fix each issue. Prose reformatting risks losing the specificity that was encoded in those fields during evaluation.

## The complete critic/reflection pipeline
The evaluate → revise loop now assembles into a single `review_and_revise` function. It runs for up to `max_iterations` cycles, stopping early when the overall score reaches the quality threshold. Two design decisions govern this loop.

First, the quality threshold (0.85 by default) is intentionally high. Critic mode is specifically for situations where output quality matters and must be verified — a low threshold would make the critic pass unnecessary. Second, the loop terminates with `ready_for_delivery=False` when max iterations are exhausted without reaching the threshold. This honest signal is more useful than claiming the output is ready when it is not: downstream systems that receive `False` can route the content for manual review rather than silently delivering substandard work.

In [9]:
def review_and_revise(
    content: str,
    criteria: list,
    context: str = "",
    quality_threshold: float = 0.85,
    max_iterations: int = 3,
) -> CritiqueResult:
    """Review content and iteratively revise until the quality threshold is met.

    Args:
        content: The content to review.
        criteria: Quality criteria to evaluate against.
        context: What the content is supposed to accomplish.
        quality_threshold: Minimum average score to accept (0.0–1.0).
        max_iterations: Maximum number of evaluate-revise cycles.

    Returns:
        CritiqueResult with the best version produced and full issue history.
    """
    print(f"\nCRITIC MODE: evaluating against {len(criteria)} criteria")
    print(f"Quality threshold: {quality_threshold:.0%} | Max iterations: {max_iterations}")
    print("=" * 60)

    current_content = content   # tracks the latest version being improved
    all_issues = []             # accumulates issues across all iterations for the audit log

    for iteration in range(1, max_iterations + 1):
        print(f"\n[Iteration {iteration}/{max_iterations}] Evaluating...")

        # Evaluate the current version (may be original or a prior revision)
        evaluation = evaluate_content(current_content, criteria, context, iteration)
        scores = evaluation["criteria_scores"]
        # Guard against an empty scores dict with a fallback of 0.0
        overall = sum(scores.values()) / len(scores) if scores else 0.0

        print(f"  Overall score: {overall:.2f} / 1.00")
        for criterion, score in scores.items():
            icon = "✓" if score >= 0.7 else "✗"
            short = criterion[:50] + "..." if len(criterion) > 50 else criterion
            print(f"  {icon} {short}: {score:.2f}")

        # Use a generator expression to convert raw dicts to typed CritiqueIssue objects
        all_issues.extend(
            CritiqueIssue(
                criterion=iss["criterion"],
                issue=iss["issue"],
                location=iss["location"],
                severity=iss["severity"],
                suggestion=iss["suggestion"],
            )
            for iss in evaluation.get("issues", [])
        )

        if overall >= quality_threshold:
            print(f"\n  \u2192 Quality threshold met at iteration {iteration}. Ready for delivery.")
            return CritiqueResult(
                original=content,
                issues_found=all_issues,
                # revised is None if no revision was needed (threshold met on first pass)
                revised=current_content if iteration > 1 else None,
                criteria_scores=scores,
                overall_score=round(overall, 2),
                iteration=iteration,
                ready_for_delivery=True,
                quality_notes=evaluation.get("overall_assessment", ""),
            )

        if iteration < max_iterations:
            print(f"  \u2192 Score {overall:.2f} below threshold {quality_threshold:.2f}. Revising...")
            current_content = revise_content(current_content, evaluation, context)
        else:
            # Max iterations exhausted — return the best version with an honest ready=False signal
            print(f"\n  \u2192 Max iterations reached. Score {overall:.2f} below threshold {quality_threshold:.2f}.")
            return CritiqueResult(
                original=content,
                issues_found=all_issues,
                revised=current_content,
                criteria_scores=scores,
                overall_score=round(overall, 2),
                iteration=iteration,
                ready_for_delivery=False,   # conservative: never claim readiness if threshold was not reached
                quality_notes=evaluation.get("overall_assessment", ""),
            )

In [10]:
# Run the full critic/reflection pipeline on the draft API documentation
result = review_and_revise(
    content=DRAFT_CONTENT,
    criteria=QUALITY_CRITERIA,
    context=CONTEXT,
    quality_threshold=0.85,
    max_iterations=3,
)

print("\n" + "=" * 60)
print("CRITIC MODE COMPLETE")
print("=" * 60)
print(f"Status        : {'READY FOR DELIVERY' if result.ready_for_delivery else 'NEEDS REVIEW'}")
print(f"Iterations    : {result.iteration}")
print(f"Overall score : {result.overall_score:.2f} / 1.00")
print(f"Total issues  : {len(result.issues_found)}")
print(f"Quality notes : {result.quality_notes}")

if result.revised:
    print("\nFinal revised content:")
    print("-" * 60)
    print(result.revised)


CRITIC MODE: evaluating against 5 criteria
Quality threshold: 85% | Max iterations: 3

[Iteration 1/3] Evaluating...
  Overall score: 0.34 / 1.00
  ✓ Accurately describes what the endpoint does and wh...: 0.70
  ✗ All parameters are documented with types, required...: 0.40
  ✗ At least one concrete request and response example...: 0.00
  ✗ Error cases and their status codes are documented: 0.00
  ✗ Is clearly understandable by a developer unfamilia...: 0.60
  → Score 0.34 below threshold 0.85. Revising...

[Iteration 2/3] Evaluating...
  Overall score: 0.74 / 1.00
  ✓ Accurately describes what the endpoint does and wh...: 1.00
  ✓ All parameters are documented with types, required...: 0.80
  ✓ At least one concrete request and response example...: 1.00
  ✗ Error cases and their status codes are documented: 0.00
  ✓ Is clearly understandable by a developer unfamilia...: 0.90
  → Score 0.74 below threshold 0.85. Revising...

[Iteration 3/3] Evaluating...
  Overall score: 0.96 / 1.00
  ✓

`all_issues.extend(...)` uses a generator expression to convert the raw dicts returned by `evaluate_content` into typed `CritiqueIssue` objects before appending them to the running list. Using `.extend` with a generator rather than appending in a loop means the conversion and collection happen in a single pass. The `all_issues` list accumulates issues across every iteration, so the returned `CritiqueResult` carries a complete audit trail of what was found at each stage — not just the issues from the final pass.

The two return paths differ in one key field: `ready_for_delivery`. The early-exit return (threshold met) passes `current_content if iteration > 1 else None` for `revised` — if no revision was ever needed, `revised` is `None`, signaling that the original content was already acceptable. The fallthrough return (max iterations exhausted) always sets `ready_for_delivery=False`, regardless of how close the score came to the threshold. This conservative design prevents downstream systems from treating a 0.84 score as acceptable when the threshold is 0.85.

## Self-reflection pattern
The maker-checker pattern above assumes two distinct roles — one agent produces, another evaluates. But a single agent can serve both roles by deliberately switching perspectives between turns. This is the self-reflection pattern.

The key to effective self-reflection is perspective switching. If the same agent evaluates its output from the same perspective it used to produce it, it will miss the same things it missed when producing. The reflection prompt below forces a perspective shift: the model is told it is a critical reviewer, not the author, and is asked to evaluate as if seeing the output for the first time. LLMs tend to be favorable to their own outputs by default — the `NOT the author` instruction and the adversarial framing ("look for what is WRONG, not what is right") counteract this tendency and produce more honest evaluations.

In [11]:
def produce_and_reflect(task: str, criteria: list, context: str = "") -> dict:
    """Produce an output, then self-critique and revise it in a single pass.

    The same agent serves as both maker (production step) and checker
    (reflection step) by explicitly switching perspective between turns.

    Args:
        task: The task to complete.
        criteria: Quality criteria for the reflection step.
        context: Context about the task and its intended audience.

    Returns:
        Dict with initial_output, critique, final_output, and revision_count.
    """
    # Step 1 — Make: production role uses a constructive, generative stance
    print("[1/3] Producing initial output (maker role)...")
    make_response = llm.invoke([
        SystemMessage(content="You are a helpful assistant. Complete the task thoroughly."),
        HumanMessage(content=f"Context: {context}\n\nTask: {task}"),
    ])
    initial_output = make_response.content.strip()
    print(f"      \u2192 Produced {len(initial_output)} characters")

    # Step 2 — Reflect: critic role uses an adversarial, evaluative stance
    # The 'NOT the author' instruction and end-user framing counteract favorable self-evaluation
    print("\n[2/3] Reflecting on output (critic role)...")
    criteria_formatted = "\n".join(f"- {c}" for c in criteria)

    reflect_response = llm.invoke([
        SystemMessage(content="""You are a critical reviewer, NOT the author.
Evaluate the output as if you were seeing it for the first time.
Look for what is WRONG, not what is right.
Evaluate from the end user's perspective, not the producer's.

Respond with JSON only:
{"needs_revision": true/false, "issues": [...], "priority_fixes": [...], "quality_score": 0.0-1.0}"""),
        HumanMessage(content=f"""Task this output was supposed to complete:
{task}

Quality criteria to evaluate against:
{criteria_formatted}

Output to review:
{initial_output}

Evaluate critically. Would an end user be fully satisfied by this output?"""),
    ])

    try:
        critique = json.loads(reflect_response.content)
    except json.JSONDecodeError:
        # Strip markdown fences if the model wrapped the JSON in a code block
        raw = reflect_response.content.strip()
        if raw.startswith("```"):
            raw = "\n".join(raw.split("\n")[1:-1])
        critique = json.loads(raw)

    print(f"      \u2192 Quality score: {critique.get('quality_score', 'N/A')}")
    print(f"      \u2192 Needs revision: {critique.get('needs_revision', False)}")

    # Step 3 — Revise: only if the critic identified issues worth fixing
    revision_count = 0
    final_output = initial_output   # default: keep the original if no revision is needed

    if critique.get("needs_revision"):
        print("\n[3/3] Revising based on self-critique...")
        priority_fixes = critique.get("priority_fixes", [])

        revise_response = llm.invoke([
            SystemMessage(content="""Revise the content to address the specific issues listed.
Fix exactly what is wrong — do not rewrite sections that are not flagged.
Return the revised content only."""),
            HumanMessage(content=f"""Original content:
{initial_output}

Issues to fix:
{json.dumps(priority_fixes, indent=2)}

Revised content:"""),
        ])
        final_output = revise_response.content.strip()
        revision_count = 1   # produce_and_reflect always does at most one revision pass
        print(f"      \u2192 Revision complete ({len(final_output)} characters)")
    else:
        print("\n[3/3] No revision needed — output meets quality criteria.")

    return {
        "task": task,
        "initial_output": initial_output,
        "critique": critique,
        "final_output": final_output,
        "revision_count": revision_count,
    }

In [12]:
# Demonstrate self-reflection: one agent produces and then critiques its own work
self_reflection_result = produce_and_reflect(
    task=(
        "Write a two-paragraph explanation of what an API rate limit is and why it exists, "
        "for a developer onboarding guide."
    ),
    criteria=[
        "Explains clearly what a rate limit is",
        "Explains why rate limits exist (purpose and benefit)",
        "Is accessible to a developer who may be encountering rate limits for the first time",
        "Is concise — two paragraphs, no unnecessary length",
    ],
    context="Developer onboarding guide for a public REST API.",
)

print("\n" + "=" * 60)
print("SELF-REFLECTION RESULT")
print("=" * 60)
print(f"Revisions made     : {self_reflection_result['revision_count']}")
print(f"Final quality score: {self_reflection_result['critique'].get('quality_score', 'N/A')}")
print("\nFinal output:")
print("-" * 60)
print(self_reflection_result["final_output"])

[1/3] Producing initial output (maker role)...
      → Produced 1308 characters

[2/3] Reflecting on output (critic role)...
      → Quality score: 0.65
      → Needs revision: True

[3/3] Revising based on self-critique...
      → Revision complete (952 characters)

SELF-REFLECTION RESULT
Revisions made     : 1
Final quality score: 0.65

Final output:
------------------------------------------------------------
An API rate limit is a rule that restricts how many requests a client can send to a server in a given time period, such as per second, minute, or hour. This limit helps ensure that all users can access the API fairly and prevents any one user from overloading the server, which could slow down or disrupt service for everyone. By enforcing rate limits, API providers maintain the API's stability and responsiveness.

Rate limits are important for several reasons. They protect server resources, allowing efficient handling of multiple requests without overload. They also help prevent

`produce_and_reflect` uses three structurally identical LLM calls but with fundamentally different system prompts. The production call uses `"You are a helpful assistant. Complete the task thoroughly."` — a constructive, generative stance. The reflection call uses `"You are a critical reviewer, NOT the author."` — an evaluative, adversarial stance. The explicit `NOT the author` instruction and the `"Look for what is WRONG, not what is right"` directive are the mechanism for perspective switching: they override the LLM's default tendency to favor its own output.

`revision_count` is a simple integer — 0 if the critique found no issues worth fixing, 1 if a revision was made. `produce_and_reflect` is designed to perform at most one revision pass because it is a single-pass self-correction pattern, not an iterative loop. When multiple revision cycles are needed, `review_and_revise` with explicit iteration control is the appropriate function.

## Avoiding common critique anti-patterns
Poorly structured critique is worse than no critique — it creates revision work without improving quality. The three most common failures are vague location (the issue is identified but not located, so the maker cannot find what to fix), global rejection (the suggestion calls for a full rewrite rather than a targeted fix), and validation without evidence (a positive assessment with no supporting citation).

The function below detects these patterns programmatically by inspecting the text of each issue's `location` and `suggestion` fields before the revision function acts on them. Detecting anti-patterns before revision prevents acting on bad feedback — a vague suggestion passed to `revise_content` will produce an equally vague revision. After the automated check, a reference table shows the full anti-pattern catalogue with symptoms and fixes so the patterns are easy to recognize when reviewing critique quality manually.

In [13]:
def detect_critique_antipatterns(issues: list) -> list:
    """Check a list of critique issues for common anti-patterns.

    Args:
        issues: List of issue dicts from evaluate_content.

    Returns:
        List of anti-pattern warning dicts found in the issue list.
    """
    warnings = []

    for issue in issues:
        # Normalize fields to lowercase for case-insensitive pattern matching
        issue_text = issue.get("issue", "").lower()
        suggestion = issue.get("suggestion", "").lower()
        location = issue.get("location", "").lower()

        # Anti-pattern 1: vague location — too broad to locate the specific problem
        vague_locations = ("throughout", "entire document", "all sections")
        if len(location) < 5 or location in vague_locations:
            warnings.append({
                "anti_pattern": "Vague location",
                "issue": issue.get("issue"),
                "problem": "Location is too vague to act on. Specify the exact section, line, or phrase.",
            })

        # Anti-pattern 2: global rejection — suggestion calls for a full rewrite rather than a fix
        rewrite_signals = ["rewrite", "completely change", "start over", "redo"]
        if any(signal in suggestion for signal in rewrite_signals):
            warnings.append({
                "anti_pattern": "Global rejection",
                "issue": issue.get("issue"),
                "problem": "Suggestion calls for a full rewrite. Identify what to preserve and what to fix.",
            })

        # Anti-pattern 3: validation without evidence — positive framing with no specific citation
        vague_positives = ["looks good", "seems fine", "appears correct", "is okay"]
        if any(pos in issue_text for pos in vague_positives):
            warnings.append({
                "anti_pattern": "Validation without evidence",
                "issue": issue.get("issue"),
                "problem": "Positive assessment has no supporting evidence. Cite specific elements.",
            })

    return warnings


# Check the issues from the initial evaluation for anti-patterns
antipatterns = detect_critique_antipatterns(evaluation.get("issues", []))

if antipatterns:
    print(f"Anti-patterns detected ({len(antipatterns)}):")
    for ap in antipatterns:
        print(f"  [{ap['anti_pattern']}] {ap['issue']}")
        print(f"    Problem: {ap['problem']}")
else:
    print("No anti-patterns detected — critique is specific and actionable.")

print("\nAnti-pattern reference:")
print("-" * 60)
antipattern_table = [
    ("Vague criticism",          "'This is unclear'",            "Specify exactly what is unclear and why"),
    ("Global rejection",         "'Needs to be rewritten'",      "Identify salvageable sections and fix specific issues"),
    ("Criteria drift",           "Adding new criteria mid-review","Define all criteria before the review begins"),
    ("Overcorrection",           "Changing style, not substance", "Stay focused on the stated criteria only"),
    ("Validation w/o evidence",  "'This looks good overall'",    "Every positive assessment must cite specific evidence"),
]
for name, symptom, fix in antipattern_table:
    print(f"  {name}")
    print(f"    Symptom: {symptom}")
    print(f"    Fix    : {fix}")
    print()

No anti-patterns detected — critique is specific and actionable.

Anti-pattern reference:
------------------------------------------------------------
  Vague criticism
    Symptom: 'This is unclear'
    Fix    : Specify exactly what is unclear and why

  Global rejection
    Symptom: 'Needs to be rewritten'
    Fix    : Identify salvageable sections and fix specific issues

  Criteria drift
    Symptom: Adding new criteria mid-review
    Fix    : Define all criteria before the review begins

  Overcorrection
    Symptom: Changing style, not substance
    Fix    : Stay focused on the stated criteria only

  Validation w/o evidence
    Symptom: 'This looks good overall'
    Fix    : Every positive assessment must cite specific evidence



`detect_critique_antipatterns` operates on plain dicts rather than `CritiqueIssue` objects because it is designed to validate critique output before it is fully parsed into the typed model. Each field is accessed with `.get()` and a default empty string to handle any missing keys gracefully — a real LLM response may occasionally omit one of the expected fields, and a `KeyError` would discard the detection result for that issue.

The `rewrite_signals` list is checked against `suggestion.lower()`, not against the issue description. This distinction matters: it is normal for an issue description to mention the word "rewrite" (e.g., "the examples section needs to be rewritten"), but when the *suggestion* calls for a full rewrite, that signals the critic has abandoned targeted feedback in favor of global rejection — which is the anti-pattern worth detecting.

- **Critic mode is for evaluation and improvement, not production**: The agent's input is existing content; its output is a revised version or a structured critique. Using critic mode to generate content from scratch defeats its purpose.
- **Quality criteria must be defined before the review begins**: Criteria drift — adding new criteria mid-review — is one of the most common anti-patterns in critique workflows. Agree on what "good" means before evaluation starts, and do not change the criteria after the first pass.
- **Specificity is the difference between useful and useless feedback**: "This section is unclear" cannot be acted on. "The third sentence in the Request section uses 'details' without specifying which fields are required" can. Every issue must have a location and a suggestion.
- **Iterative revision with a quality threshold is more reliable than a single pass**: One pass catches obvious issues. Multiple passes with a measurable threshold catch the issues that the first revision introduced or missed.
- **Revision is constrained, not open-ended**: Fix what is flagged; do not rewrite what is not. Unrequested rewrites introduce new errors and scope changes that the critique did not authorize.